# 🎙️ За гранью: озвучка 28 сцен

Бесплатная пакетная генерация русской речи с Silero V5 CIS Base. **Не нужен платный API-ключ**. Используется открытая модель в среде Colab. Запускать клетки по порядку кнопками ▶. По окончании будет скачан ZIP с WAV-файлами.

**Важно:** голоса Silero могут быть менее эмоциональны, чем HeyGen; эффект делает их мрачнее. Время работы и доступность бесплатного Colab зависят от Google.

**Исправление:** используется проверенная русская модель `v5_ru`, голос `aidar` (вместо некорректного имени голоса).


In [ ]:
!pip -q install silero soundfile scipy requests
import os, json, re, io, zipfile, requests, numpy as np, torch
from scipy import signal
from scipy.io import wavfile
from IPython.display import display, Audio
from pathlib import Path
BASE='https://raw.githubusercontent.com/sanya377/Za_graniy/main/audio/narration-scripts.json'
res=requests.get(BASE,timeout=30);res.raise_for_status()
project=res.json(); scenes=project['files']
print('✅ Загружено сцен:',len(scenes))
print('Для теста:',scenes[0]['title'])


In [ ]:
from silero import silero_tts
# Русская модель с автоматической расстановкой ударений.
# В предыдущей версии было недопустимое имя голоса ru_bel_anatoliy.
model, example_text = silero_tts(language='ru', speaker='v5_ru')
SPEAKER='aidar'  # штатный русский мужской голос модели v5_ru
SAMPLE_RATE=24000
print('✅ Модель загружена. Голос:',SPEAKER)
print('Доступные голоса:',model.speakers)
assert SPEAKER in model.speakers, 'Голос отсутствует в модели'


In [ ]:
def chunks(text, max_len=340):
    text=re.sub(r'<[^>]*>','',text).strip()
    parts=re.split(r'(?<=[.!?…])\s+', text)
    out=[];current=''
    for p in parts:
        if len(current)+len(p)+1<=max_len:
            current=(current+' '+p).strip()
        else:
            if current:out.append(current)
            while len(p)>max_len:
                cut=p.rfind(' ',0,max_len)
                if cut<50:cut=max_len
                out.append(p[:cut]);p=p[cut:].lstrip()
            current=p
    if current:out.append(current)
    return out

def mystical(audio, sr=SAMPLE_RATE, dark=.40):
    # Мягкий низкий тембр, чуть замедленная речь, лёгкое реверберационное эхо.
    a=np.asarray(audio,dtype=np.float32).reshape(-1)
    a=signal.resample_poly(a,100,104)  # ~4% более глубокое звучание
    sos=signal.butter(2, min(5000,sr*.44), btype='lowpass', fs=sr,output='sos')
    a=signal.sosfilt(sos,a).astype(np.float32)
    echo=int(sr*.16);out=np.pad(a,(0,int(sr*.85)))
    if len(a)>echo:out[echo:echo+len(a)]+=(.11*dark)*a
    echo2=int(sr*.31)
    if len(a)>echo2:out[echo2:echo2+len(a)]+=(.07*dark)*a
    peak=max(np.max(np.abs(out)),1e-6)
    return (out/peak*.86).astype(np.float32)

def voice_for_scene(scene):
    # Единый рассказчик сохраняет целостность повествования.
    # Изменить персонажей и голоса можно будет после первого прослушивания.
    parts=[]
    for chunk in chunks(scene['text']):
        a=model.apply_tts(text=chunk,speaker=SPEAKER,sample_rate=SAMPLE_RATE)
        parts.append(np.asarray(a,dtype=np.float32).reshape(-1))
        parts.append(np.zeros(int(SAMPLE_RATE*.48),dtype=np.float32))
    return mystical(np.concatenate(parts))

def save_wav(name, audio):
    Path('narration').mkdir(exist_ok=True)
    wavfile.write('narration/'+name+'.wav',SAMPLE_RATE,(np.clip(audio,-1,1)*32767).astype(np.int16))


## ▶ Сначала послушай пролог
После выполнения появится встроенный проигрыватель. Если голос подходит, запускай следующую клетку.


In [ ]:
print('Синтезирую пролог…')
intro=voice_for_scene(scenes[0])
save_wav('start',intro)
print('🎧 Пролог:',scenes[0]['title'])
display(Audio(intro,rate=SAMPLE_RATE))


## ▶ Озвучить все 28 сцен и скачать архив
Файлы будут созданы в папке `narration/`, затем собраны в ZIP. Готовые звуковые дорожки можно загрузить в GitHub.


In [ ]:
import time
for index,scene in enumerate(scenes,1):
    dest=Path('narration')/(scene['id']+'.wav')
    if dest.exists():
        print(f'{index:02d}/{len(scenes)} ✅ {scene["id"]} (уже готово)')
        continue
    try:
        track=voice_for_scene(scene)
        save_wav(scene['id'],track)
        print(f'{index:02d}/{len(scenes)} ✅ {scene["title"]}')
    except Exception as e:
        print(f'{index:02d}/{len(scenes)} ❌ {scene["id"]}: {e}')
files=sorted(Path('narration').glob('*.wav'))
archive='/content/za_graniy_ozvuchka.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in files:z.write(p,'assets/voice/'+p.name)
print(f'📦 Готово {len(files)} из {len(scenes)} сцен. Архив:',archive)
if len(files)==len(scenes):
    from google.colab import files as colab_files
    colab_files.download(archive)
else:
    print('Часть сцен не создалась. Прочитай ошибки и перезапусти эту клетку.')
